In [8]:
import os
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm

# Directories
INPUT_DIR = "D:/Rock_CT_images/1A-135k-Data"       # Directory containing original grayscale images
OUTPUT_DIR = "D:/Rock_CT_images/Sliding_Window"    # Directory to save crops & labels
CROP_SIZE = 128                   # Crop size (128x128)
STRIDE = 64                        # Stride for sliding window (Overlap = CROP_SIZE - STRIDE)
x1, y1 = 280, 110   # Top-left corner
x2, y2 = 1000, 820  # Bottom-right corner


# Load all images from the directory
image_files = [f for f in os.listdir(INPUT_DIR) if f.endswith((".jpg", ".png", ".jpeg"))]

# Process each image
for image_file in tqdm(image_files, desc="Processing Images"):
    img_path = os.path.join(INPUT_DIR, image_file)

    # Load image in grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    img = img[y1:y2, x1:x2]
    if img is None:
        continue  # Skip if the image can't be loaded

    h, w = img.shape

    # Slide the window over the image
    crop_count = 0
    for y in range(0, h - CROP_SIZE + 1, STRIDE):
        for x in range(0, w - CROP_SIZE + 1, STRIDE):
            # Extract the 128x128 crop
            crop = img[y:y + CROP_SIZE, x:x + CROP_SIZE]
            
            original_img_pil = Image.fromarray(crop)

            # Save images
            crop_filename = f"{image_file.split('.')[0]}_{crop_count}.png"
            original_img_pil.save(f"{OUTPUT_DIR}/{crop_filename}")

            crop_count += 1

print("Data preparation complete! Cropped & masked images saved.")


Processing Images: 100%|██████████| 49/49 [00:08<00:00,  5.83it/s]

Data preparation complete! Cropped & masked images saved.


In [15]:
import os
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm

# 📂 Input and output directories
INPUT_DIR = "D:/Rock_CT_images/Sliding_Window"    # Directory containing grayscale images
OUTPUT_DIR = "D:/Rock_CT_images/Self-Supervised"  # Directory to save masked images & labels
MASK_RATIO = 0.2              # Percentage of patches to be masked
MASK_SIZE = 5

# Ensure output directories exist
os.makedirs(f"{OUTPUT_DIR}/masked", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels", exist_ok=True)


In [16]:
# 🔹 Load all images from the directory
image_files = [f for f in os.listdir(INPUT_DIR) if f.endswith((".jpg", ".png", ".jpeg"))]

# 🔥 Process each image
for image_file in tqdm(image_files, desc="Processing Images"):
    img_path = os.path.join(INPUT_DIR, image_file)

    # Load image in grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue  # Skip if the image can't be loaded

    h, w = img.shape
    masked_img = img.copy()  # Copy for masking

    # Calculate number of blocks to mask
    num_blocks = int((h * w * MASK_RATIO) / (MASK_SIZE * MASK_SIZE))

    for _ in range(num_blocks):
        # Randomly select top-left corner of the mask
        x = np.random.randint(0, w - MASK_SIZE)
        y = np.random.randint(0, h - MASK_SIZE)
        
        # Apply black mask (block masking)
        masked_img[y:y+MASK_SIZE, x:x+MASK_SIZE] = 0

    # Save images
    cv2.imwrite(f"{OUTPUT_DIR}/masked/{image_file}", masked_img)  # Masked image (input)
    cv2.imwrite(f"{OUTPUT_DIR}/labels/{image_file}", img)  # Original image (label)

print("✅ Data preparation complete! Masked images & labels saved.")

Processing Images: 100%|██████████| 4900/4900 [00:06<00:00, 768.93it/s]

✅ Data preparation complete! Masked images & labels saved.
